**Eksperyment 1: Wpływ funkcji aktywacji na aproksymację funkcji w zadaniu regresji**
**Testowane Layery:**
- 0 hidden-layers, 16 perceptronów per layer
- 1 hidden-layers, 16 perceptronów per layer
- 2 hidden-layers,, 16 perceptronów per layer, 16-32, 32-32
- 3 hidden-layers, 16 perceptronów per layer
- 4 hidden-layers, 16 perceptronów per layer


**Każdy eksperyment będzie miał różną architektóre sieci, ale stałe parametry to:**
- aktywacja wyjściowa to f(x) = x
- funkcja aktywacji to sigmoid
- funkcja straty to MSE
- optymalizatior to SGD
- learning rate = 0.005
- ilość epok = 500
- batch = 32


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

from mlp_model.activations.activation import Activation
from mlp_model.activations.relu import ReLU
from mlp_model.activations.swish import Swish
from mlp_model.layers.layer import Layer

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from mlp_model.activations.sigmoid import Sigmoid
from mlp_model.initialize_weights.xavier import XavierInitializer
from mlp_model.layers.hidden import Hidden
from mlp_model.layers.output_layer import OutputLayer
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.mlp import MLP
from mlp_model.optimizers.sgd import SGDOptimizer
from trainer.trainer import MLPTrainer
from sklearn.model_selection import train_test_split
from visualisation.regression_visualise import plot_learning_curve, plot_regression_fit, \
    plot_weights_evolution, print_cv_summary

data_kind = "activation"
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/regression" / f"data.{data_kind}.train.{data_count}.csv"
df = pd.read_csv(train_filepath)
x = df["x"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
y = df["y"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)

C:\Users\arczi\PycharmProjects\NeuralNetworks\visualize_plot\helo_helo.png


In [2]:
def layers_cv(l: list[Layer], filename: str = None):
    mlp = MLP(layers=l)
    mlp.initialize_weights(XavierInitializer())
    sgd = SGDOptimizer(mlp, 0.005)
    trainer = MLPTrainer(mlp, sgd, MSELoss(), 100, 32)
    summary = trainer.cross_validate(x, y, 5)
    print_cv_summary(summary, filename)

def visualize_layer(l: list[Layer], filename: str = None):
    mlp = MLP(layers=l)
    mlp.initialize_weights(XavierInitializer())
    sgd = SGDOptimizer(mlp, 0.005)
    trainer = MLPTrainer(mlp, sgd, MSELoss(), 100, 32)
    X_train, X_val, y_train, y_val = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
    )
    trainer.fit(X_train, y_train, X_val, y_val)
    plot_learning_curve(trainer.history, filename)
    plot_regression_fit(trainer.history, mlp, X_train, y_train, X_val, y_val, filename)
    for i in range(len(l)):
        plot_weights_evolution(trainer.history, i,  16, filename)

def test_layers(l: list[Layer], filename: str = None):
    layers_cv(l, filename)
    visualize_layer(l, filename)

In [3]:
layers = {
    "0": [OutputLayer(1,1)],
    "1": [Hidden(1,16, Sigmoid()), OutputLayer(16,1)],
    "2-8-16": [Hidden(1,8, Sigmoid()), Hidden(8,16, Sigmoid()), OutputLayer(16,1)],
    "2-16-16": [Hidden(1,16, Sigmoid()), Hidden(16,16, Sigmoid()), OutputLayer(16,1)],
    "2-16-32": [Hidden(1,16, Sigmoid()), Hidden(16,32, Sigmoid()), OutputLayer(32,1)],
    "2-32-32": [Hidden(1,32, Sigmoid()), Hidden(32,32, Sigmoid()), OutputLayer(32,1)],
    "3": [Hidden(1,16, Sigmoid()), Hidden(16,16, Sigmoid()), Hidden(16,16, Sigmoid()), OutputLayer(16,1)],
    "4": [Hidden(1,16, Sigmoid()), Hidden(16,16, Sigmoid()), Hidden(16,16, Sigmoid()), Hidden(16,16, Sigmoid()), OutputLayer(16,1)],
}

In [4]:
for k, v in layers.items():
    print(f"--------| {k} |--------")
    filename = f"/regression/architecture/{k}/layers"
    test_layers(v, filename)

--------| 0 |--------
Fold 1: loss = 672.090900 | metrics {'r2': 0.9119969358454376, 'mae': 23.099872115086747, 'mse': 672.0908998180062}
Fold 2: loss = 691.171565 | metrics {'r2': 0.9103261796196469, 'mae': 23.552810119761443, 'mse': 691.1715647437777}
Fold 3: loss = 721.570967 | metrics {'r2': 0.9032195919006394, 'mae': 24.04497698146101, 'mse': 721.570967306112}
Fold 4: loss = 756.572745 | metrics {'r2': 0.909467886617539, 'mae': 24.763198103387158, 'mse': 756.5727451275204}
Fold 5: loss = 702.946128 | metrics {'r2': 0.915936580990644, 'mae': 23.754575494167646, 'mse': 702.9461279101051}
---------------------------------------------------------------
Cross validation with 5 folds:
Mean loss: 708.870461 (std: 28.767675)
Mean metrics:
  R2: 0.910189 (std: 0.004133)
  MAE: 23.843087 (std: 0.553503)
  MSE: 708.870461 (std: 28.767675)
---------------------------------------------------------------
--------| 1 |--------
Fold 1: loss = 0.205026 | metrics {'r2': 0.9999731540398774, 'mae': 0